In [11]:
import numpy as np
import pandas as pd
from scipy import stats

# 1. Select metric type here ("std" or "ci")
STAT_TYPE = "std"  # Change to "ci" whenever you want Confidence Intervals


def summarize_metric(group, column, stat_type=STAT_TYPE, confidence=0.95):
    """Summarizes a column as Mean ± [Std | CI] (raw_values)."""
    values = group[column].dropna().to_numpy(dtype=float)
    n = len(values)

    if n == 0:
        return "N/A"

    mean = np.mean(values)

    if n > 1:
        if stat_type == "std":
            margin = np.std(values, ddof=1)
        elif stat_type == "ci":
            sem = stats.sem(values)
            margin = sem * stats.t.ppf((1 + confidence) / 2, df=n - 1)
        else:
            raise ValueError("stat_type must be either 'std' or 'ci'")

        margin_str = f"{margin:.2f}"
    else:
        margin_str = "N/A"

    actual = ", ".join(f"{x:.2f}" for x in values)

    return f"{mean:.2f} ± {margin_str} ({actual})"




from pathlib import Path
import json

import numpy as np
import pandas as pd

def actual_sampler(args):
    requested = args.get("sampler", "auto")

    if requested != "auto":
        return requested

    objective = args.get("objective")

    defaults = {
        "ddpm": "ddim",
        "edm": "heun",
        "flow_matching": "euler",
    }

    return defaults.get(objective, "unknown")

def summarize_run(run_dir, tail_n=5):
    run_dir = Path(run_dir)

    with open(run_dir / "args.json", encoding="utf-8") as f:
        args = json.load(f)

    fid_path = run_dir / "validation_fid.jsonl"
    fids = []

    if fid_path.exists():
        with open(fid_path, encoding="utf-8") as f:
            fids = [
                json.loads(line)
                for line in f
                if line.strip()
            ]

    row = {
        "run": run_dir.name,
        **{f"arg_{key}": value for key, value in args.items()},
        "actual_sampler": actual_sampler(args),
        "fid_best": np.nan,
        "fid_best_iteration": np.nan,
        "fid_final": np.nan,
        "fid_final_iteration": np.nan,
        "fid_tail_median": np.nan,
        "fid_tail_mean": np.nan,
        "fid_tail_max": np.nan,
        "fid_tail_n": 0,
        "run_full": str(run_dir),
    }

    if fids:
        values = np.array([item["fid"] for item in fids], dtype=float)
        iterations = np.array([item["iteration"] for item in fids])

        tail = values[-tail_n:]

        best_index = np.argmin(values)

        row.update({
            "fid_best": values[best_index],
            "fid_best_iteration": iterations[best_index],
            "fid_final": values[-1],
            "fid_final_iteration": iterations[-1],
            "fid_tail_median": np.median(tail),
            "fid_tail_mean": np.mean(tail),
            "fid_tail_max": np.max(tail),
            "fid_tail_n": len(tail),
        })

    return row



def summarize_runs(experiment_dirs, tail_n=5):
    """Summarizes all run subdirectories found inside multiple experiment directories."""
    rows = []

    # Handle both a single path string/Path object and a list of paths
    if isinstance(experiment_dirs, (str, Path)):
        experiment_dirs = [experiment_dirs]

    for exp_dir in experiment_dirs:
        exp_path = Path(exp_dir)

        # Iterate through subdirectories inside each experiment root
        for run_dir in exp_path.iterdir():
            # Check if it's a directory and contains args.json
            if run_dir.is_dir() and (run_dir / "args.json").exists():
                rows.append(summarize_run(run_dir, tail_n=tail_n))

    return pd.DataFrame(rows)

# Example usage:


In [19]:
# 2. Build comparison DataFrame
suffix = "std" if STAT_TYPE == "std" else "95ci"

dirs = [
    "/home/satoshi/projects/fcmstylegan/experiments/eeeg/diffusion/",
    "/home/satoshi/projects/fcmstylegan/experiments/diffusion/phase1_compact/",
]
df = summarize_runs(dirs)

df = df[df['actual_sampler']!='ddim']


comparison = (
    df.groupby(
        [   
            "arg_batch",
            "arg_seed",
            "arg_profile_encoder",
            "arg_backbone",
            "arg_objective",
            "actual_sampler",
            "fid_final_iteration",
            # "run_full"
            
        ],
        dropna=False,
    )
    .apply(
        lambda g: pd.Series({
            "runs": len(g),
            f"tail_fid_{suffix}": summarize_metric(g, "fid_tail_median", stat_type=STAT_TYPE),
            f"best_fid_{suffix}": summarize_metric(g, "fid_best", stat_type=STAT_TYPE),
            f"final_fid_{suffix}": summarize_metric(g, "fid_final", stat_type=STAT_TYPE),
            f"tail_max_fid_{suffix}": summarize_metric(g, "fid_tail_max", stat_type=STAT_TYPE),
        })
    )
)

display(comparison)

runs  \
arg_batch arg_seed arg_profile_encoder arg_backbone arg_objective actual_sampler fid_final_iteration         
128       123      cnn                 adm          edm           heun           275000                  1   
                                       compact      ddpm          ddpm           195000                  1   
                   mlp                 adm          edm           heun           295000                  1   
                                       compact      ddpm          ddpm           300000                  1   

                                                                                                             tail_fid_std  \
arg_batch arg_seed arg_profile_encoder arg_backbone arg_objective actual_sampler fid_final_iteration                        
128       123      cnn                 adm          edm           heun           275000               17.53 ± N/A (17.53)   
                                       compact      ddpm          ddpm           195000               13.51 ± N/A (13.51)   
                   mlp                 adm          edm           heun           295000               15.09 ± N/A (15.09)   
                                       compact      ddpm          ddpm           300000               12.72 ± N/A (12.72)   

                                                                                                             best_fid_std  \
arg_batch arg_seed arg_profile_encoder arg_backbone arg_objective actual_sampler fid_final_iteration                        
128       123      cnn                 adm          edm           heun           275000               11.88 ± N/A (11.88)   
                                       compact      ddpm          ddpm           195000               12.46 ± N/A (12.46)   
                   mlp                 adm          edm           heun           295000               11.92 ± N/A (11.92)   
                                       compact      ddpm          ddpm           300000               12.41 ± N/A (12.41)   

                                                                                                            final_fid_std  \
arg_batch arg_seed arg_profile_encoder arg_backbone arg_objective actual_sampler fid_final_iteration                        
128       123      cnn                 adm          edm           heun           275000               16.12 ± N/A (16.12)   
                                       compact      ddpm          ddpm           195000               13.23 ± N/A (13.23)   
                   mlp                 adm          edm           heun           295000               15.28 ± N/A (15.28)   
                                       compact      ddpm          ddpm           300000               13.00 ± N/A (13.00)   

                                                                                                         tail_max_fid_std  
arg_batch arg_seed arg_profile_encoder arg_backbone arg_objective actual_sampler fid_final_iteration                       
128       123      cnn                 adm          edm           heun           275000               21.16 ± N/A (21.16)  
                                       compact      ddpm          ddpm           195000               13.71 ± N/A (13.71)  
                   mlp                 adm          edm           heun           295000               15.28 ± N/A (15.28)  
                                       compact      ddpm          ddpm           300000               13.00 ± N/A (13.00)

- `backbone`: The `compact` or `adm` U-Net choice affects the network architecture, not the meaning of the objective or sampler.
- `objective` describes how the model is trained.  
- `sampler` describes how images are generated after training.

| Objective | Model learns to predict | Compatible samplers |
|---|---|---|
| `ddpm` | Added Gaussian noise `ε` | `ddpm`, `ddim` |
| `edm` | EDM-preconditioned denoising residual | `euler`, `heun` |
| `flow_matching` | Velocity from noise to image | `euler`, `heun` |

There is no `ddim` objective because DDIM is not a training objective. It is a sampling algorithm for a model trained with the DDPM objective.

For example:

```bash
--objective ddpm --sampler ddim
```

means:

1. Train the model to predict noise using DDPM.
2. Generate images using the DDIM sampling trajectory.

Similarly:

```bash
--objective flow_matching --sampler heun
```

means:

1. Train the model to predict flow velocity.
2. Integrate that velocity using Heun’s method.



# experiment plans
- do not use `ddim` or `euler` as they are just approximation to make inference faster. then we only have 3 x 3 = 9 options
- `backbone` has three choices {`compact`,`adm`,`dit`}. 
- (`objective`,`sampler`) also has three choices {(`ddpm`,`ddpm`),(`edm`,`heun`),(`flow_matching`,`heun`)}
- Default three choices (`backbone`,`objective`,`sampler`) should be 
    - (`compact`,`ddpm`,`ddpm`)
    - (`adm`,`edm`,`heun`)
    - (`dit`,`flow_matching`,`heun`)



In [29]:
from itertools import product
import shlex

python = "python"
gpu = "3"

datasplit = "./data/task1_dataset_split.csv"
preprocessed_root = "/dev/shm/satoshi.tsutsui/data/task1_processed"
sweep_root = "experiments/diffusion_sweep"

seeds = range(4)

backbones = {
    "dit": {
        "script": "train_jit.py",
        "args": ["--model", "JiT-B/16"],
    },
    "compact": {
        "script": "train_diffusion.py",
        "args": ["--backbone", "compact"],
    },
    "adm": {
        "script": "train_diffusion.py",
        "args": ["--backbone", "adm"],
    },
}

objective_sampler_pairs = [
    ("ddpm", "ddpm"),
    ("edm", "heun"),
    ("flow_matching", "heun"),
]

common_args = [
    "--datasplit", datasplit,
    "--preprocessed_root", preprocessed_root,
    "--batch", "128",
    "--bf16",
    "--compile_mode", "default",
    "--fid_samples", "1000",
]

commands = []

for seed, (backbone, (objective, sampler)) in product(
    seeds,
    product(backbones, objective_sampler_pairs),
):
    config_name = f"{backbone}_{objective}_seed{seed}"
    exp_dir = f"{sweep_root}"

    command_parts = [
        python,
        backbones[backbone]["script"],
        *common_args,
        *backbones[backbone]["args"],
        "--objective", objective,
        "--sampler", sampler,
        "--seed", str(seed),
        "--exp_dir", exp_dir,
    ]

    commands.append(" ".join(shlex.quote(part) for part in command_parts))

output_path = "../exp_diffusion_sweep.yaml"
output_path = Path(output_path)
with output_path.open("w", encoding="utf-8") as file:
    for command in commands:
        file.write(f"- cmd: {command}\n")

In [ ]:
# 2. Build comparison DataFrame
suffix = "std" if STAT_TYPE == "std" else "95ci"

dirs = [
    "/home/satoshi/projects/fcmstylegan/experiments/eeeg/diffusion_sweep/",
    "/home/satoshi/projects/fcmstylegan/experiments/diffusion/jit/",
]
df = summarize_runs(dirs)

df = df[df['actual_sampler']!='ddim']


comparison = (
    df.groupby(
        [   
            "arg_batch",
            "arg_seed",
            "arg_profile_encoder",
            "arg_backbone",
            "arg_objective",
            "actual_sampler",
            "fid_final_iteration",
            # "run_full"
            
        ],
        dropna=False,
    )
    .apply(
        lambda g: pd.Series({
            "runs": len(g),
            f"tail_fid_{suffix}": summarize_metric(g, "fid_tail_median", stat_type=STAT_TYPE),
            f"best_fid_{suffix}": summarize_metric(g, "fid_best", stat_type=STAT_TYPE),
            f"final_fid_{suffix}": summarize_metric(g, "fid_final", stat_type=STAT_TYPE),
            f"tail_max_fid_{suffix}": summarize_metric(g, "fid_tail_max", stat_type=STAT_TYPE),
        })
    )
)

display(comparison)